<a href="https://colab.research.google.com/github/Org-Space-Medicine-Engineering-Design/project-lunar/blob/main/notebooks/02_BR_UTMB_descriptive_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — BR_UTMB Descriptive Analysis

Descriptive analysis of the **UTMB Bed Rest** dataset: demographics + standard biochemical profile across timepoints (BDC / HDT / R+).

**Pipeline**
1. Load CSVs from `../data/raw/BR_UTMB/`
2. Inspect dimensions
3. Identify variable names and dtypes
4. Demographic summaries
5. Biochemical summary statistics (overall + per timepoint)
6. Missingness — by variable, by participant, by timepoint
7. Units & naming inconsistencies
8. Export summary tables

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

# Paths — adjust if you're running locally vs in Colab
REPO       = Path('.').resolve()
DATA_DIR   = REPO / 'data' / 'raw' / 'BR_UTMB'
OUT_STATS  = REPO / 'outputs' / 'descriptive_stats'
OUT_MISS   = REPO / 'outputs' / 'missingness'
OUT_STATS.mkdir(parents=True, exist_ok=True)
OUT_MISS.mkdir(parents=True, exist_ok=True)

DATASET = 'BR_UTMB'
print(f'Repo root : {REPO}')
print(f'Data dir  : {DATA_DIR}')

## Column classifier

Heuristics to bucket columns into **id / timepoint / demographic / biochemical / other**. Override in `MANUAL_OVERRIDES` if auto-detection misses anything in your CSVs.

In [ ]:
# Manual overrides — fill in if heuristics miss anything specific to your CSVs.
MANUAL_OVERRIDES: dict[str, str] = {}

ID_PATTERNS        = [r'^subject', r'^subj', r'^pid$', r'_id$', r'^id$', r'participant', r'^crew']
TIMEPOINT_PATTERNS = [r'visit', r'timepoint', r'time_?point', r'^session', r'phase', r'^day$', r'study_?day',
                      r'^week$', r'bdc', r'hdt', r'recovery', r'^r\+', r'pre_?post']
DEMO_PATTERNS = {
    'age'   : [r'^age', r'_age$'],
    'sex'   : [r'^sex$', r'gender'],
    'bmi'   : [r'^bmi$', r'body_?mass_?index'],
    'height': [r'^height', r'^ht_?cm', r'stature'],
    'weight': [r'^weight', r'^wt_?kg', r'^body_?mass'],
    'race'  : [r'race', r'ethnic'],
}

def classify_column(col, dtype):
    if col in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[col]
    c = col.lower().strip()
    if any(re.search(p, c) for p in ID_PATTERNS):
        return 'id'
    if any(re.search(p, c) for p in TIMEPOINT_PATTERNS):
        return 'timepoint'
    for _, pats in DEMO_PATTERNS.items():
        if any(re.search(p, c) for p in pats):
            return 'demographic'
    if pd.api.types.is_numeric_dtype(dtype):
        return 'biochemical'
    return 'other'

def classify_frame(df):
    buckets = {'id': [], 'timepoint': [], 'demographic': [], 'biochemical': [], 'other': []}
    for c in df.columns:
        buckets[classify_column(c, df[c].dtype)].append(c)
    return buckets

## 1 · Load CSV files

Reads every `*.csv` in `data/raw/BR_UTMB/`. Each file → its own DataFrame keyed by stem.

In [ ]:
csv_paths = sorted(DATA_DIR.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSVs found in {DATA_DIR}. Drop your UTMB files there and re-run.')

frames = {}
for p in csv_paths:
    try:
        df = pd.read_csv(p)
    except UnicodeDecodeError:
        df = pd.read_csv(p, encoding='latin-1')
    frames[p.stem] = df
    print(f'  ✓ {p.name:<40s}  {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols')

print(f'\nLoaded {len(frames)} file(s).')

## 2 · Inspect dimensions

In [ ]:
dim_rows = []
for name, df in frames.items():
    buckets = classify_frame(df)
    id_col  = buckets['id'][0] if buckets['id'] else None
    tp_col  = buckets['timepoint'][0] if buckets['timepoint'] else None
    dim_rows.append({
        'file'         : name,
        'rows'         : df.shape[0],
        'cols'         : df.shape[1],
        'id_col'       : id_col,
        'n_participants': df[id_col].nunique() if id_col else np.nan,
        'timepoint_col': tp_col,
        'n_timepoints' : df[tp_col].nunique() if tp_col else np.nan,
    })
dims = pd.DataFrame(dim_rows)
dims

## 3 · Variable names & data types


In [ ]:
var_rows = []
for name, df in frames.items():
    for col in df.columns:
        var_rows.append({
            'file'    : name,
            'column'  : col,
            'dtype'   : str(df[col].dtype),
            'role'    : classify_column(col, df[col].dtype),
            'n_unique': df[col].nunique(dropna=True),
            'n_null'  : df[col].isna().sum(),
            'pct_null': round(df[col].isna().mean() * 100, 2),
        })
var_dict = pd.DataFrame(var_rows)
print(f'{len(var_dict)} variables across {len(frames)} file(s)')
var_dict.head(30)

## 4 · Build the analytic table

If multiple CSVs share an ID (and optionally a timepoint), they get outer-merged into one long table called `df_all`. If only one CSV exists it's used directly.

In [ ]:
def primary_keys(df):
    b = classify_frame(df)
    return b['id'][:1] + b['timepoint'][:1]

if len(frames) == 1:
    df_all = next(iter(frames.values())).copy()
else:
    items = list(frames.items())
    df_all = items[0][1].copy()
    base_keys = primary_keys(df_all)
    for name, df in items[1:]:
        keys = [k for k in primary_keys(df) if k in base_keys and k in df.columns]
        if keys:
            df_all = df_all.merge(df, on=keys, how='outer', suffixes=('', f'__{name}'))
            print(f'  merged {name} on {keys}')
        else:
            print(f'  ⚠ skipped {name} — no shared id/timepoint keys with base ({base_keys})')

buckets = classify_frame(df_all)
ID_COL = buckets['id'][0]  if buckets['id'] else None
TP_COL = buckets['timepoint'][0] if buckets['timepoint'] else None
print(f'\nAnalytic table: {df_all.shape[0]:,} rows × {df_all.shape[1]} cols')
print(f'ID column      : {ID_COL}')
print(f'Timepoint col  : {TP_COL}')
df_all.head()

## 5 · Demographic summaries

In [ ]:
demo_cols = buckets['demographic']
print('Demographic columns detected:', demo_cols)

# Collapse to one row per participant
if ID_COL and demo_cols:
    demo_df = df_all.groupby(ID_COL)[demo_cols].first().reset_index()
else:
    demo_df = df_all[demo_cols].copy() if demo_cols else pd.DataFrame()

n_participants = demo_df[ID_COL].nunique() if ID_COL else len(demo_df)
n_timepoints   = df_all[TP_COL].nunique() if TP_COL else 1
print(f'\nN participants : {n_participants}')
print(f'N timepoints   : {n_timepoints}')
if TP_COL:
    print('\nObservations per timepoint:')
    print(df_all[TP_COL].value_counts().sort_index().to_string())

In [ ]:
def numeric_summary(s):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if s.empty:
        return {k: np.nan for k in ['n', 'mean', 'sd', 'median', 'q1', 'q3', 'iqr', 'min', 'max']}
    q1, q3 = s.quantile([0.25, 0.75])
    return {
        'n'     : int(s.size),
        'mean'  : s.mean(),
        'sd'    : s.std(),
        'median': s.median(),
        'q1'    : q1,
        'q3'    : q3,
        'iqr'   : q3 - q1,
        'min'   : s.min(),
        'max'   : s.max(),
    }

def categorical_summary(s):
    counts = s.value_counts(dropna=False)
    pct = (counts / counts.sum() * 100).round(2)
    return pd.DataFrame({'n': counts, 'pct': pct})

demo_numeric_rows, demo_categorical_blocks = [], {}
for col in demo_cols:
    s = demo_df[col]
    if pd.api.types.is_numeric_dtype(s):
        row = {'variable': col, **numeric_summary(s)}
        demo_numeric_rows.append(row)
    else:
        demo_categorical_blocks[col] = categorical_summary(s)

demo_numeric = pd.DataFrame(demo_numeric_rows)
demo_numeric

In [ ]:
for col, tbl in demo_categorical_blocks.items():
    print(f'\n— {col} —')
    print(tbl.to_string())

## 6 · Biochemical summary statistics

Overall summary across all observations, plus per-timepoint when a timepoint column exists.

In [ ]:
biochem_cols = buckets['biochemical']
print(f'{len(biochem_cols)} biochemical variables detected')

biochem_overall = pd.DataFrame(
    [{'variable': c, **numeric_summary(df_all[c])} for c in biochem_cols]
)
biochem_overall.head(20)

In [ ]:
if TP_COL and biochem_cols:
    long_rows = []
    for tp, sub in df_all.groupby(TP_COL):
        for c in biochem_cols:
            long_rows.append({'timepoint': tp, 'variable': c, **numeric_summary(sub[c])})
    biochem_by_tp = pd.DataFrame(long_rows)
    print(f'Biochem × timepoint table: {biochem_by_tp.shape}')
    display(biochem_by_tp.head(20))
else:
    biochem_by_tp = pd.DataFrame()
    print('No timepoint column — skipping per-timepoint biochemistry summary.')

## 7 · Missingness assessment

Three lenses — by variable, by participant, by participant × timepoint.

In [ ]:
miss_by_var = (
    df_all.isna()
          .mean()
          .mul(100).round(2)
          .rename('pct_missing')
          .to_frame()
          .assign(n_missing=df_all.isna().sum())
          .sort_values('pct_missing', ascending=False)
)
miss_by_var.head(30)

In [ ]:
if ID_COL:
    miss_by_subj = (
        df_all.drop(columns=[c for c in [ID_COL, TP_COL] if c])
              .isna()
              .groupby(df_all[ID_COL])
              .mean()
              .mean(axis=1)
              .mul(100).round(2)
              .rename('pct_missing')
              .sort_values(ascending=False)
              .to_frame()
    )
    display(miss_by_subj.head(20))
else:
    miss_by_subj = pd.DataFrame()
    print('No ID column — skipping per-participant missingness.')

In [ ]:
if ID_COL and TP_COL:
    miss_by_subj_tp = (
        df_all.drop(columns=[ID_COL, TP_COL])
              .isna()
              .groupby([df_all[ID_COL], df_all[TP_COL]])
              .mean()
              .mean(axis=1)
              .mul(100).round(2)
              .rename('pct_missing')
              .reset_index()
    )
    display(miss_by_subj_tp.head(20))
else:
    miss_by_subj_tp = pd.DataFrame()

## 8 · Units & naming inconsistencies

Flags columns that might be the same variable spelled differently, and tries to pull units out of column names.

In [ ]:
UNIT_PAT = re.compile(r'(?:[\(\[_\s-])([a-zA-Zµμ]+(?:[/\.][a-zA-Zµμ]+)?\d*)(?:[\)\]_\s-]|$)')

KNOWN_UNITS = {'mg', 'ng', 'pg', 'g', 'kg', 'mmol', 'umol', 'µmol', 'nmol', 'iu', 'miu',
               'mg/dl', 'g/dl', 'mmol/l', 'umol/l', 'nmol/l', 'pg/ml', 'ng/ml', 'iu/l',
               'meq/l', 'pct', '%', 'k/ul', 'm/ul', 'cells/ul'}

def extract_unit(col):
    for m in UNIT_PAT.finditer(col):
        cand = m.group(1).lower()
        if cand in KNOWN_UNITS:
            return cand
    return None

def normalize(col):
    return re.sub(r'[\s_\-\(\)\[\]\./]+', '', col.lower())

name_groups = {}
for col in df_all.columns:
    name_groups.setdefault(normalize(col), []).append(col)
dupes = {k: v for k, v in name_groups.items() if len(v) > 1}

print(f'Potential naming collisions (same variable, different spelling): {len(dupes)}')
for k, v in list(dupes.items())[:20]:
    print(f'  {k:<30s} → {v}')

units_table = pd.DataFrame(
    [{'column': c, 'inferred_unit': extract_unit(c)} for c in df_all.columns]
)
units_with = units_table.dropna(subset=['inferred_unit'])
print(f'\nColumns with inferable units: {len(units_with)} / {len(df_all.columns)}')
units_with.head(20)

## 9 · Export summary tables

Writes all results to CSV plus a consolidated Excel workbook with one sheet per table.

In [ ]:
exports = {
    OUT_STATS / f'{DATASET}_dimensions.csv'            : dims,
    OUT_STATS / f'{DATASET}_variable_dictionary.csv'   : var_dict,
    OUT_STATS / f'{DATASET}_demographics_numeric.csv'  : demo_numeric,
    OUT_STATS / f'{DATASET}_biochem_overall.csv'       : biochem_overall,
    OUT_STATS / f'{DATASET}_biochem_by_timepoint.csv'  : biochem_by_tp,
    OUT_STATS / f'{DATASET}_units_inferred.csv'        : units_table,
    OUT_MISS  / f'{DATASET}_missing_by_variable.csv'   : miss_by_var,
    OUT_MISS  / f'{DATASET}_missing_by_participant.csv': miss_by_subj,
    OUT_MISS  / f'{DATASET}_missing_by_subj_x_tp.csv'  : miss_by_subj_tp,
}
for path, tbl in exports.items():
    if tbl is None or (hasattr(tbl, 'empty') and tbl.empty):
        print(f'  ⊘ skipped (empty) {path.name}')
        continue
    tbl.to_csv(path, index=isinstance(tbl.index, pd.MultiIndex) or tbl.index.name is not None)
    print(f'  ✓ wrote {path.relative_to(REPO)}')

# Consolidated Excel workbook — this is THE deliverable spreadsheet
xlsx_path = OUT_STATS / f'{DATASET}_summary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as xl:
    dims.to_excel(xl, sheet_name='dimensions', index=False)
    var_dict.to_excel(xl, sheet_name='variable_dictionary', index=False)
    demo_numeric.to_excel(xl, sheet_name='demo_numeric', index=False)
    biochem_overall.to_excel(xl, sheet_name='biochem_overall', index=False)
    if not biochem_by_tp.empty:
        biochem_by_tp.to_excel(xl, sheet_name='biochem_by_timepoint', index=False)
    miss_by_var.to_excel(xl, sheet_name='missing_by_variable')
    if not miss_by_subj.empty:
        miss_by_subj.to_excel(xl, sheet_name='missing_by_participant')
    if not miss_by_subj_tp.empty:
        miss_by_subj_tp.to_excel(xl, sheet_name='missing_by_subj_x_tp', index=False)
    units_table.to_excel(xl, sheet_name='units_inferred', index=False)

print(f'\n✓ Consolidated workbook → {xlsx_path.relative_to(REPO)}')
print('  This is the spreadsheet deliverable. Upload it to outputs/descriptive_stats/ on GitHub.')